In [7]:
# ============================================================
# Fourier coefficient recovery under noise and subsampling
# ============================================================
#
# This script generates Figure 1 for the paper:
#   Section 4.1 "Recovery of known spectral structure"
#
# Purpose
# -------
# The experiment evaluates whether sparse regression recovers the known
# Fourier structure of the function
#
#     f(x) = x,   x in [-pi, pi],
#
# whose sine-series coefficients satisfy
#
#     |b_k| = 2 / k.
#
# The script compares:
#   (i) ordinary least squares (OLS), and
#   (ii) sparse regression via LASSO (used here as the SOR estimator),
#
# under additive Gaussian noise and limited training data.
#
# Experimental setup
# ------------------
# - A uniform grid of N points is generated on [-pi, pi].
# - The target values are y = f(x) plus Gaussian noise with standard
#   deviation sigma.
# - A random subset of 20% of the samples is used for training.
# - The remaining 80% is used as a held-out test set.
# - Both OLS and LASSO are fit in a trigonometric design matrix
#   [1, sin(x), cos(x), sin(2x), cos(2x), ...].
#
# Outputs
# -------
# The script saves two figures:
#
#   1a.pdf  Magnitudes of the recovered sine coefficients |b_k|
#           compared with the analytic decay 2/k.
#
#   1b.pdf  Function reconstruction evaluated on the held-out test set,
#           together with the noisy training samples.
#
# Connection to the paper
# -----------------------
# This experiment supports the claims made in:
#
# - Section 3, Theorem 1:
#   bounded-variation functions admit compressible Fourier expansions
#   with coefficient decay proportional to 1/k.
#
# - Section 3, Theorem 2:
#   sparse regression in an orthogonal / near-orthogonal design suppresses
#   unstable coefficients under noise.
#
# In the paper, Figure 1 is used to show that:
# - OLS fails to recover the correct spectral decay under noise and
#   subsampling, producing unstable high-frequency coefficients.
# - Sparse regression recovers the dominant low-frequency structure and
#   yields a more stable reconstruction on held-out data.
#
# Notes
# -----
# - The design here is a standard trigonometric basis, not an explicitly
#   normalized orthonormal basis.
# - The LASSO penalty is fixed rather than tuned to enforce a target number
#   of nonzero coefficients.
# - Test RMSE is computed on clean held-out targets.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso
import os

# ============================================================
# Output directory
# ============================================================
FIGDIR = "figs"
os.makedirs(FIGDIR, exist_ok=True)

# ============================================================
# Global plotting style for all figures
# ============================================================
PLOT_STYLE = {
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "lines.linewidth": 2.2,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "axes.linewidth": 1.1,
    "savefig.format": "pdf",
}

AXIS_LABEL_SIZE = PLOT_STYLE["axes.labelsize"]
TITLE_SIZE = PLOT_STYLE["axes.titlesize"]
TICK_LABEL_SIZE = PLOT_STYLE["xtick.labelsize"]
LEGEND_SIZE = PLOT_STYLE["legend.fontsize"]
LINE_WIDTH = PLOT_STYLE["lines.linewidth"]

plt.rcParams.update(PLOT_STYLE)

# ============================================================
# Trigonometric design matrix on [-pi, pi]
# Column order:
#   0: 1
#   1: sin(1x), 2: cos(1x),
#   3: sin(2x), 4: cos(2x), ...
# ============================================================
def trig_design(x, K):
    x = np.asarray(x)
    N = len(x)
    Phi = np.ones((N, 2 * K + 1))
    col = 1
    for k in range(1, K + 1):
        Phi[:, col] = np.sin(k * x)
        col += 1
        Phi[:, col] = np.cos(k * x)
        col += 1
    return Phi

# ============================================================
# True Fourier sine coefficients for f(x)=x on [-pi,pi]
# x = 2 sum_{k>=1} (-1)^{k+1} sin(kx)/k
# so:
#   b_k = 2 (-1)^{k+1}/k
# ============================================================
def true_bk(K):
    k = np.arange(1, K + 1, dtype=float)
    return 2.0 * ((-1.0) ** (k + 1.0)) / k

def extract_bk_from_coef(coef, K):
    # coef layout: [1, sin1, cos1, sin2, cos2, ...]
    return np.array([coef[2 * k - 1] for k in range(1, K + 1)])

def apply_common_axis_style(ax):
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=TICK_LABEL_SIZE,
        length=PLOT_STYLE["xtick.major.size"],
        width=PLOT_STYLE["xtick.major.width"],
    )
    ax.xaxis.label.set_size(AXIS_LABEL_SIZE)
    ax.yaxis.label.set_size(AXIS_LABEL_SIZE)
    ax.title.set_size(TITLE_SIZE)

# ============================================================
# Main experiment: noise + 20% training subset
# Reconstruction is shown on the held-out 80% test set
# ============================================================
def run_noisy_subsampled_fourier_example(
    N=400,
    train_frac=0.2,
    sigma=0.15,
    K=30,
    lasso_alpha=3e-3,
    seed=0
):
    rng = np.random.default_rng(seed)

    # Domain and function
    a_dom, b_dom = -np.pi, np.pi
    f = lambda x: x

    # Full dataset
    x_all = np.linspace(a_dom, b_dom, N)
    y_clean = f(x_all)
    y_noisy = y_clean + sigma * rng.standard_normal(size=N)

    # Random split
    n_train = int(train_frac * N)
    all_idx = np.arange(N)
    train_idx = rng.choice(N, size=n_train, replace=False)
    test_idx = np.setdiff1d(all_idx, train_idx)

    # Training data (noisy)
    x_train = x_all[train_idx]
    y_train = y_noisy[train_idx]

    # Test data (clean target for evaluation)
    x_test = x_all[test_idx]
    y_test = y_clean[test_idx]

    # Sort test points for plotting
    sort_test = np.argsort(x_test)
    x_test_plot = x_test[sort_test]
    y_test_plot = y_test[sort_test]

    # Design matrices
    Phi_train = trig_design(x_train, K)
    Phi_test = trig_design(x_test, K)
    Phi_test_plot = trig_design(x_test_plot, K)

    # -----------------------
    # OLS
    # -----------------------
    ols = LinearRegression(fit_intercept=False)
    ols.fit(Phi_train, y_train)
    yhat_ols_test = Phi_test @ ols.coef_
    yhat_ols_plot = Phi_test_plot @ ols.coef_

    # -----------------------
    # SOR (LASSO)
    # -----------------------
    sor = Lasso(alpha=lasso_alpha, fit_intercept=False, max_iter=300000)
    sor.fit(Phi_train, y_train)
    yhat_sor_test = Phi_test @ sor.coef_
    yhat_sor_plot = Phi_test_plot @ sor.coef_

    # -----------------------
    # Coefficients (sine only)
    # -----------------------
    b_true = true_bk(K)
    b_ols = extract_bk_from_coef(ols.coef_, K)
    b_sor = extract_bk_from_coef(sor.coef_, K)
    k_vals = np.arange(1, K + 1)

    # ========================================================
    # FIGURE A: sine coefficient magnitudes |b_k|
    # ========================================================
    figA, axA = plt.subplots(figsize=(10, 5))
    axA.plot(k_vals, np.abs(b_true), "k-o", label=r"True $|b_k| = 2/k$")
    axA.plot(k_vals, np.abs(b_ols), "s--", label="OLS")
    axA.plot(k_vals, np.abs(b_sor), "d--", label="SOR")
    axA.set_yscale("log")
    axA.set_xlabel("Fourier Mode")
    axA.set_ylabel(r"Sine Coef. Magnitude $|b_k|$")
    axA.set_title(
        rf"Fourier Sine Coeff. for $f(x)=x$ "
        rf"(20\% train, 80\% test, noise $\sigma = {sigma}$)"
    )
    axA.legend(fontsize=LEGEND_SIZE)
    apply_common_axis_style(axA)
    figA.tight_layout()
    figA.savefig(os.path.join(FIGDIR, "1a.pdf"), bbox_inches="tight")
    plt.close(figA)

    # ========================================================
    # FIGURE B: reconstruction on held-out test data
    # ========================================================
    figB, axB = plt.subplots(figsize=(10, 5))
    axB.plot(x_test_plot, y_test_plot, "k", lw=LINE_WIDTH, label=r"True $f(x)=x$ (test)")
    axB.plot(x_test_plot, yhat_ols_plot, "--", lw=LINE_WIDTH, label="OLS Reconstruction")
    axB.plot(x_test_plot, yhat_sor_plot, "-.", lw=LINE_WIDTH, label="SOR Reconstruction")

    # Optional: show training samples used for fitting
    axB.scatter(
        x_train, y_train,
        s=18, alpha=0.65, marker="o",
        label="Noisy training samples"
    )

    axB.set_xlabel("Input $x$")
    axB.set_ylabel(r"Function Value")
    axB.set_title(
        rf"Reconstruction on held-out test data "
        rf"(20% train, 80% test, noise $\sigma = {sigma}$)"
    )
    axB.legend(fontsize=LEGEND_SIZE)
    apply_common_axis_style(axB)
    figB.tight_layout()
    figB.savefig(os.path.join(FIGDIR, "1b.pdf"), bbox_inches="tight")
    plt.close(figB)

    # Test RMSE
    rmse_ols = float(np.sqrt(np.mean((yhat_ols_test - y_test) ** 2)))
    rmse_sor = float(np.sqrt(np.mean((yhat_sor_test - y_test) ** 2)))
    nnz_sor = int(np.count_nonzero(np.abs(sor.coef_) > 1e-10))

    print(f"Saved figures to ./{FIGDIR}/")
    print(f"N={N}, train={n_train} ({train_frac*100:.0f}%), test={N-n_train} ({100-train_frac*100:.0f}%), sigma={sigma}, K={K}")
    print(f"Test RMSE: OLS={rmse_ols:.3e} | SOR={rmse_sor:.3e} | SOR nnz={nnz_sor}/{len(sor.coef_)}")

if __name__ == "__main__":
    run_noisy_subsampled_fourier_example(
        N=400,
        train_frac=0.2,
        sigma=0.15,
        K=30,
        lasso_alpha=3e-3,
        seed=0
    )

Saved figures to ./figs/
N=400, train=80 (20%), test=320 (80%), sigma=0.15, K=30
Test RMSE: OLS=4.803e+00 | SOR=7.869e-01 | SOR nnz=50/61
